In [1]:
import os
import gc
import psutil

from utils.postgres_util import (
    get_engine_from_env, 
    read_sql_dataframe
)

from utils.synthetic.pipeline.melt_stage_writer import (
    build_sensor_messages_stage,
    validate_sensor_messages_stage,
)



In [3]:
SCHEMA = str("capstone")

DATASET_ID = str("pump_synthetic_v1")
RUN_ID = str("premelt_run_001")

IF_EXISTS_FLAG = str("replace")
RANDOM_SEED = int(42)
NUMBER_OF_SENSORS = int(52)
CHUNK_SIZE = int(10000)

MELT_SOURCE_TABLE_NAME = str("synthetic_observations_timestamped_stage")
MELT_DESTINATION_TABLE_NAME = str("synthetic_sensor_messages_stage")


---

In [2]:

def memory_gb() -> float:
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 ** 3)

def log_memory(label: str) -> None:
    print(f"[memory] {label}: {memory_gb():.2f} GB")

---

In [4]:

engine = get_engine_from_env()


---

In [5]:
log_memory("before read")
# read chunk
log_memory("after read")
# melt chunk
log_memory("after melt")
# write chunk
log_memory("after write")
gc.collect()
log_memory("after gc")

[memory] before read: 0.17 GB
[memory] after read: 0.17 GB
[memory] after melt: 0.17 GB
[memory] after write: 0.17 GB
[memory] after gc: 0.17 GB


---

In [6]:

melt_table_name = build_sensor_messages_stage(
    engine=engine,
    schema=SCHEMA,
    source_table=MELT_SOURCE_TABLE_NAME,
    target_table=MELT_DESTINATION_TABLE_NAME,
    if_exists=IF_EXISTS_FLAG,
    random_seed=RANDOM_SEED,
    n_sensors=NUMBER_OF_SENSORS,
    chunk_size=CHUNK_SIZE,
)


[chunk] 1 | source rows 0 to 9,999
[chunk] 1 melted 10,000 observations -> 520,000 sensor rows
[chunk] 1 wrote 520,000 rows to capstone.synthetic_sensor_messages_stage
[chunk] 2 | source rows 10,000 to 19,999
[chunk] 2 melted 10,000 observations -> 520,000 sensor rows
[chunk] 2 wrote 520,000 rows to capstone.synthetic_sensor_messages_stage
[chunk] 3 | source rows 20,000 to 29,999
[chunk] 3 melted 10,000 observations -> 520,000 sensor rows
[chunk] 3 wrote 520,000 rows to capstone.synthetic_sensor_messages_stage
[chunk] 4 | source rows 30,000 to 39,999
[chunk] 4 melted 10,000 observations -> 520,000 sensor rows
[chunk] 4 wrote 520,000 rows to capstone.synthetic_sensor_messages_stage
[chunk] 5 | source rows 40,000 to 49,999
[chunk] 5 melted 10,000 observations -> 520,000 sensor rows
[chunk] 5 wrote 520,000 rows to capstone.synthetic_sensor_messages_stage
[chunk] 6 | source rows 50,000 to 59,999
[chunk] 6 melted 10,000 observations -> 520,000 sensor rows
[chunk] 6 wrote 520,000 rows to cap

---

In [7]:
log_memory("before read")
# read chunk
log_memory("after read")
# melt chunk
log_memory("after melt")
# write chunk
log_memory("after write")
gc.collect()
log_memory("after gc")

[memory] before read: 0.62 GB
[memory] after read: 0.62 GB
[memory] after melt: 0.62 GB
[memory] after write: 0.62 GB
[memory] after gc: 0.62 GB


---

In [8]:

print("Built table:", melt_table_name)


Built table: synthetic_sensor_messages_stage


---

In [9]:

validation_dataframe = validate_sensor_messages_stage(
    engine=engine,
    schema=SCHEMA,
    table_name=MELT_DESTINATION_TABLE_NAME,
)

display(validation_dataframe)

,row_count,distinct_observation_count,distinct_observation_timestamp_count,distinct_sensor_name_count,min_sensor_index,max_sensor_index,min_message_sequence_index,max_message_sequence_index
0,3744000,72000,72000,52,0,51,0,51


---

In [10]:
sample_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        observation_index,
        observation_timestamp,
        generated_row_id,
        sensor_name,
        sensor_index,
        sensor_value,
        message_sequence_index,
        stream_state,
        phase,
        meta_episode_id
    FROM capstone.synthetic_sensor_messages_stage
    ORDER BY observation_index, message_sequence_index
    LIMIT 104
    """
)

display(sample_dataframe)

,observation_index,observation_timestamp,generated_row_id,sensor_name,sensor_index,sensor_value,message_sequence_index,stream_state,phase,meta_episode_id
0,1,2026-03-19 08:00:00+00:00,premelt_run_001_obs_000000000001,sensor_41,41,44.118690,0,normal,normal,0
1,1,2026-03-19 08:00:00+00:00,premelt_run_001_obs_000000000001,sensor_49,49,59.773464,1,normal,normal,0
2,1,2026-03-19 08:00:00+00:00,premelt_run_001_obs_000000000001,sensor_46,46,48.795368,2,normal,normal,0
3,1,2026-03-19 08:00:00+00:00,premelt_run_001_obs_000000000001,sensor_32,32,571.445862,3,normal,normal,0
4,1,2026-03-19 08:00:00+00:00,premelt_run_001_obs_000000000001,sensor_18,18,3.253774,4,normal,normal,0
...,...,...,...,...,...,...,...,...,...,...
99,2,2026-03-19 08:01:00+00:00,premelt_run_001_obs_000000000002,sensor_13,13,11.127496,47,normal,normal,0
100,2,2026-03-19 08:01:00+00:00,premelt_run_001_obs_000000000002,sensor_11,11,49.728981,48,normal,normal,0
101,2,2026-03-19 08:01:00+00:00,premelt_run_001_obs_000000000002,sensor_07,7,15.269876,49,normal,normal,0
102,2,2026-03-19 08:01:00+00:00,premelt_run_001_obs_000000000002,sensor_08,8,15.255219,50,normal,normal,0


---

In [11]:
sequence_check_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        observation_index,
        COUNT(*) AS message_count,
        COUNT(DISTINCT sensor_index) AS distinct_sensor_count,
        COUNT(DISTINCT observation_timestamp) AS distinct_observation_timestamp_count,
        MIN(message_sequence_index) AS min_msg_seq,
        MAX(message_sequence_index) AS max_msg_seq,
        COUNT(DISTINCT message_sequence_index) AS distinct_msg_seq_count
    FROM capstone.synthetic_sensor_messages_stage
    GROUP BY observation_index
    ORDER BY observation_index
    LIMIT 10
    """
)

display(sequence_check_dataframe)


,observation_index,message_count,distinct_sensor_count,distinct_observation_timestamp_count,min_msg_seq,max_msg_seq,distinct_msg_seq_count
0,1,52,52,1,0,51,52
1,2,52,52,1,0,51,52
2,3,52,52,1,0,51,52
3,4,52,52,1,0,51,52
4,5,52,52,1,0,51,52
5,6,52,52,1,0,51,52
6,7,52,52,1,0,51,52
7,8,52,52,1,0,51,52
8,9,52,52,1,0,51,52
9,10,52,52,1,0,51,52


----

In [12]:
# Dispose Engine
engine.dispose()